# Verified $D_4$ and Klein-quartic period models

This notebook constructs genuine principally polarized abelian varieties from normalized period matrices, derives their Riemann metrics, verifies the polarization conditions, and computes the type-$(2,\ldots,2)$ GKP relative systoles.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == 'notebooks' else cwd
src = repo_root / 'src'
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from gkp_systole import (
    D4_PERIOD_MODEL,
    KLEIN_QUARTIC_PERIOD_MODEL,
    VERIFIED_PERIOD_MODELS,
    validate_d4_derivation,
)

print(f'Repository: {repo_root}')

Repository: .


## Normalized period matrices

We write $\Lambda=\mathbb Z^g+\Omega\mathbb Z^g$. The $D_4$ period matrix is derived from the covolume-one symplectic $D_4$ lattice. The Klein matrix is the one in Bochnak--Kucharz--Silhol, Example 4.16.

In [2]:
for model in VERIFIED_PERIOD_MODELS:
    print('\n' + model.name)
    print('source:', model.source)
    print('CM field:', model.cm_field)
    print('Omega ~=')
    for row in model.period_numeric:
        print('   ', tuple(f'{value.real:.9f}{value.imag:+.9f}i' for value in row))


D4 principally polarized abelian surface
source: Derived in this repository from the covolume-one symplectic D4 lattice; Conway--Sloane, appendix to Buser--Sarnak (1994), DOI 10.1007/BF01232233.
CM field: Q(sqrt(-2))
Omega ~=
    ('0.500000000+0.707106781i', '0.500000000+0.000000000i')
    ('0.500000000+0.000000000i', '0.500000000+0.707106781i')

Jacobian of the Klein quartic
source: Bochnak--Kucharz--Silhol (1997), Example 4.16, Publications Mathématiques de l'IHÉS 86, pp. 61--62.
CM field: Q(sqrt(-7))
Omega ~=
    ('0.500000000+0.566946710i', '0.000000000+0.377964473i', '0.000000000+0.377964473i')
    ('0.000000000+0.377964473i', '0.500000000+0.566946710i', '0.000000000+0.377964473i')
    ('0.000000000+0.377964473i', '0.000000000+0.377964473i', '0.500000000+0.566946710i')


## Exact Riemann checks

For each model the code checks:

1. $X=X^T$ and $Y=Y^T>0$;
2. the principal alternating form has type $(1,\ldots,1)$;
3. the derived Riemann metric has covolume one;
4. the induced complex structure satisfies $I^2=-1$ exactly.

For $D_4$ it additionally verifies the integral symplectic basis change from the standard root-lattice Gram matrix.

In [3]:
validate_d4_derivation()
print('D4 root-lattice derivation: PASS')
for model in VERIFIED_PERIOD_MODELS:
    model.validate()
    print(f'{model.name}: Riemann checks PASS')

D4 root-lattice derivation: PASS
D4 principally polarized abelian surface: Riemann checks PASS
Jacobian of the Klein quartic: Riemann checks PASS


## Metrics derived from the periods

For $\Omega=X+iY$, the lattice-coordinate metric is

$$G_\Omega=\begin{pmatrix}Y^{-1}&Y^{-1}X\\XY^{-1}&XY^{-1}X+Y\end{pmatrix}.$$

Both examples factor as $G=G_{\rm core}/\sqrt r$, leaving a rational positive-definite core for certified shortest-vector enumeration.

In [4]:
for model in VERIFIED_PERIOD_MODELS:
    print(f'\n{model.name}: G = G_core/sqrt({model.scale_radicand})')
    for row in model.metric_core:
        print('   ', tuple(str(value) for value in row))


D4 principally polarized abelian surface: G = G_core/sqrt(2)
    ('2', '0', '1', '1')
    ('0', '2', '1', '1')
    ('1', '1', '2', '1')
    ('1', '1', '1', '2')

Jacobian of the Klein quartic: G = G_core/sqrt(28)
    ('20', '-8', '-8', '10', '-4', '-4')
    ('-8', '20', '-8', '-4', '10', '-4')
    ('-8', '-8', '20', '-4', '-4', '10')
    ('10', '-4', '-4', '8', '0', '0')
    ('-4', '10', '-4', '0', '8', '0')
    ('-4', '-4', '10', '0', '0', '8')


## Certified type-$(2,\ldots,2)$ GKP results

In [5]:
results = {}
print(f'{"model":42} {"type":11} {"exact ell^2":20} {"ell^2":>14} {"ell":>14} {"Nc":>4} {"Nl":>4}')
print('-' * 124)
for model in VERIFIED_PERIOD_MODELS:
    result = model.compute_qubit_systole()
    results[model.name] = result
    print(
        f'{model.name:42} '
        f'{str(result.core_result.polarization.type):11} '
        f'{result.squared_systole_expression:20} '
        f'{result.squared_systole:14.12f} '
        f'{result.systole:14.12f} '
        f'{result.class_multiplicity:4d} '
        f'{result.lift_multiplicity:4d}'
    )
    assert result.certified

d4 = results[D4_PERIOD_MODEL.name]
klein = results[KLEIN_QUARTIC_PERIOD_MODEL.name]
assert abs(d4.squared_systole - 1/(2*2**0.5)) < 1e-12
assert (d4.class_multiplicity, d4.lift_multiplicity) == (12, 24)
assert abs(klein.squared_systole - 1/7**0.5) < 1e-12
assert (klein.class_multiplicity, klein.lift_multiplicity) == (21, 42)

model                                      type        exact ell^2                   ell^2            ell   Nc   Nl
----------------------------------------------------------------------------------------------------------------------------
D4 principally polarized abelian surface   (2, 2)      (1/2)/sqrt(2)        0.353553390593 0.594603557501   12   24
Jacobian of the Klein quartic              (2, 2, 2)   1/sqrt(7)            0.377964473009 0.614788152951   21   42


## Interpretation

The values above are no longer standalone root-lattice checks. Each metric is derived from an explicit normalized period matrix, and the principal Riemann form, covolume, and complex-structure compatibility are verified before the type-$(2,\ldots,2)$ relative systole is computed.